# About
This noteook is for processing the data in the CSV file that can be found (website [here](https://www.briancjenkins.com/dmp-model/)). This notebook reads in the csv file with macroeconomic data (which has already been partially preprocessed by others) and computes the necessary features needed to calibrate the toy DMP model they recommended.

In [ ]:
### Import in the data (preprocessed, took the CSV off the website)
import pandas as pd
from pathlib import Path

df_path = Path("data/real/beveridge_curve_data.csv") # change filepath as needed
df = pd.read_csv(df_path)
df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)
print(df.head())

            Unemployment [Thousands of persons]  \
Date                                              
1929-04-01                               346.61   
1929-05-01                               829.57   
1929-06-01                              1036.60   
1929-07-01                               397.87   
1929-08-01                                20.16   

            Labor force [Thousands of persons]  \
Date                                             
1929-04-01                            50233.47   
1929-05-01                            50276.86   
1929-06-01                            50320.26   
1929-07-01                            50363.66   
1929-08-01                            50407.05   

            Vacancies [Thousands of vacancies]  
Date                                            
1929-04-01                             1007.22  
1929-05-01                              980.47  
1929-06-01                              962.91  
1929-07-01                              989.13

In [12]:
### Compute necessary features for calibration
vacancy_series_unscaled = df["Vacancies [Thousands of vacancies]"]
labor_force_series = df["Labor force [Thousands of persons]"]

# Construct the vacancy_rate series
vacancy_rate_series = vacancy_series_unscaled / labor_force_series

# Scale the vacanry rates to ensure that the average vacancy rate for 1965 is
#  0.0205 to match Zagorsky (1998) vacancy rate estimate
vacancy_rate_1965 = vacancy_rate_series[
    (vacancy_rate_series.index>=pd.to_datetime('1965-01-01'))
    & 
    (vacancy_rate_series.index<=pd.to_datetime('1965-12-01'))
    ]

scaling = vacancy_rate_1965.mean() / 0.0205

vacancy_rate_series = 100 * vacancy_rate_series / scaling

# define some derivative series
vacancy_series = vacancy_rate_series*labor_force_series/100
unemployment_series = df["Unemployment [Thousands of persons]"]
unemployment_rate_series = unemployment_series / labor_force_series
market_tightness_series = vacancy_series / unemployment_series

In [ ]:
### join the time series together + save it
df_rates = pd.concat(
    [unemployment_rate_series,vacancy_rate_series,market_tightness_series]
    , join='outer'
    , axis = 1
    ).dropna()
df_rates.columns = ['Unemployment rate','Vacancy rate','Market tightness']

In [ ]:
### optional: save the fully processed table
import os

folderpath = Path(os.getcwd()) / 'data' / "real" # change filepath as needed
outputfile = folderpath / 'cleaned_dmp_data.csv'
df.to_csv(outputfile, index=True)

In [ ]:
### Estimate model parameters